# Phase A — BGE-M3 Sanity Check (Colab A100)

**목적**: 로컬 `vector_service.py`의 BGE-M3 세팅이 silent failure 3가지 중 어느 것에 해당하는지 검증.

## 검증 항목
1. **Pooling mode** — BGE-M3 권장은 `CLS`. `mean` 이면 retrieval 품질 저하.
2. **max_seq_length** — 512 이면 본문 silent truncation. 원본 BGE-M3은 8192까지 지원.
3. **Embedding dimension** — 1024 여야 ChromaDB collection(`_BGE_M3_DIM = 1024`)과 호환.
4. **Query discrimination** — 관련 없는 쿼리 간 similarity 평균 < 0.5 여야 건전.

**실행 순서**: 위→아래로 셀 순차 실행. Cell 6의 진단 요약이 최종 OK/NG.

## Cell 1 — 환경 체크 및 의존성 설치

In [ ]:
!nvidia-smi | head -20
import subprocess
subprocess.run(["pip", "install", "-q",
                "sentence-transformers==3.1.1",
                "transformers",
                "torch",
                "numpy"], check=True)

import torch
print(f"\n=== 환경 ===")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("!! GPU 미탐지 - Runtime > Change runtime type > GPU (A100) 선택 필요")

## Cell 2 — BGE-M3 로드 및 세팅 검사

In [ ]:
from sentence_transformers import SentenceTransformer

print("모델 다운로드 시작 (BAAI/bge-m3, ~2.2GB)...")
model = SentenceTransformer("BAAI/bge-m3", device="cuda")
print("로드 완료.\n")

transformer_module = model[0]
pooling_module = model[1]

print("=== 모델 구조 ===")
print(f"model class: {type(model).__name__}")
print(f"layers: {[type(m).__name__ for m in model]}")
print(f"max_seq_length: {model.max_seq_length}")
print(f"embedding dim: {model.get_sentence_embedding_dimension()}")

print("\n=== Pooling 세부 ===")
pooling_mode = pooling_module.get_pooling_mode_str()
print(f"pooling mode (string): {pooling_mode}")
print(f"  cls_token:    {pooling_module.pooling_mode_cls_token}")
print(f"  mean_tokens:  {pooling_module.pooling_mode_mean_tokens}")
print(f"  max_tokens:   {pooling_module.pooling_mode_max_tokens}")
print(f"  weightedmean: {getattr(pooling_module, 'pooling_mode_weightedmean_tokens', False)}")

## Cell 3 — max_seq 실험 (truncation 탐지)

In [ ]:
short = "삼성전자 2024년 매출액은 약 300조원입니다."
long_text = ("삼성전자 반도체 사업부의 분기별 매출과 영업이익 추이 및 DRAM NAND 가격 변동 분석 " * 300).strip()

tokens_short = model.tokenizer.encode(short, add_special_tokens=True)
tokens_long_full = model.tokenizer.encode(long_text, add_special_tokens=True, truncation=False)
tokens_long_trunc = model.tokenizer.encode(long_text, add_special_tokens=True, truncation=True, max_length=model.max_seq_length)

print(f"short text tokens:        {len(tokens_short)}")
print(f"long text tokens (full):  {len(tokens_long_full)}")
print(f"long text tokens (trunc): {len(tokens_long_trunc)}")
print(f"model max_seq_length:     {model.max_seq_length}")
print()

if len(tokens_long_full) > model.max_seq_length:
    dropped = len(tokens_long_full) - model.max_seq_length
    print(f"[RED] SILENT TRUNCATION: long_text 의 {dropped} 토큰이 silent drop")
    print("   -> 실제 인덱싱 시 긴 본문은 뒤쪽이 잘립니다.")
else:
    print("[GREEN] truncation 없음 — 본문 전체 인덱싱 가능")

emb_short = model.encode(short, normalize_embeddings=True)
emb_long = model.encode(long_text, normalize_embeddings=True)
print(f"\nembedding shape short: {emb_short.shape}")
print(f"embedding shape long:  {emb_long.shape}")

## Cell 4 — 로컬 파이프라인 재현 (`build_embedding_text` 기준)

In [ ]:
# embedding_strategy.py::build_embedding_text 로직 재현
def build_embedding_text(header_dict, body):
    lines = [f"{k}: {v}" for k, v in header_dict.items() if v]
    header = "\n".join(lines)
    return f"{header}\n\n본문:\n{body}"

sample_header = {
    "문서유형": "사업보고서",
    "회사명": "삼성전자",
    "공시종류": "정기공시",
    "세부유형": "재무제표/주석",
    "공시일": "2024-03-29",
    "보고기간": "2023년 연간",
    "섹션": "연결재무제표 주석 제22. 유동부채 - 단기차입금 차입금 세부내역",
    "핵심태그": "유동성, 단기차입금, 매출채권, 운전자본, 운영자금",
}
sample_body = (
    "주식회사 삼성전자 및 종속기업은 2023년 12월 31일 현재 단기차입금을 다음과 같이 보유하고 있습니다. "
    "무역금융, 운영자금, 일반대출 등 목적별 구분으로 각 금융기관과의 계약 조건을 상세히 기술하고 있으며 "
    "이자율 범위는 연 1.2%에서 7.8% 수준입니다. "
) * 100

full_text = build_embedding_text(sample_header, sample_body)
header_only = build_embedding_text(sample_header, "")

tokens_full = model.tokenizer.encode(full_text, add_special_tokens=True, truncation=False)
tokens_header = model.tokenizer.encode(header_only, add_special_tokens=True, truncation=False)
body_tokens = len(tokens_full) - len(tokens_header)

print(f"헤더만 tokens:        {len(tokens_header)}")
print(f"헤더+본문 tokens:     {len(tokens_full)}")
print(f"본문 tokens:          ~{body_tokens}")
print(f"model max_seq_length: {model.max_seq_length}")
print()

if len(tokens_full) > model.max_seq_length:
    budget_for_body = model.max_seq_length - len(tokens_header)
    coverage = budget_for_body / body_tokens * 100 if body_tokens else 0
    print(f"[RED] 실제 재무 청크 truncation 발생!")
    print(f"   본문 {body_tokens} tokens 중 {budget_for_body} tokens 만 반영 (coverage {coverage:.1f}%)")
    print(f"   나머지 {body_tokens - budget_for_body} tokens silent drop")
    print(f"\n   -> **대책**: max_seq 확장 or 청크 크기 축소 or 헤더 압축")
else:
    remaining = model.max_seq_length - len(tokens_full)
    print(f"[GREEN] 전체 {len(tokens_full)} tokens <= max_seq {model.max_seq_length} (여유 {remaining} tokens)")

## Cell 5 — Query Discrimination Sanity

In [ ]:
import numpy as np

queries = [
    "삼성전자 2024년 매출액",
    "SK하이닉스 영업이익률 2024년",
    "2025년 상장폐지 사유 top5",
    "감사의견 부적정 받은 기업",
    "자본잠식 발생 기업 top10",
    "반도체 업황 전망",
    "현대차 배당 정책 2024",
    "삼성바이오로직스 CMO 수주",
    "카카오 플랫폼 규제 리스크",
    "LG에너지솔루션 배터리 공급계약",
]

embs = model.encode(queries, normalize_embeddings=True, convert_to_numpy=True)
sim = embs @ embs.T

print("=== Query-Query Similarity Matrix ===")
print("(off-diagonal 값이 낮을수록 의미 구분 잘 되는 상태)\n")

header = "      " + "  ".join([f"Q{i+1:2d}" for i in range(len(queries))])
print(header)
for i, q in enumerate(queries):
    row = "  ".join([f"{sim[i][j]:.2f}" for j in range(len(queries))])
    print(f"Q{i+1:2d}:  {row}  | {q[:32]}")

off_diag = [sim[i][j] for i in range(len(queries)) for j in range(len(queries)) if i != j]
mean_off = float(np.mean(off_diag))
max_off = float(np.max(off_diag))
min_off = float(np.min(off_diag))

print(f"\n평균 off-diagonal similarity: {mean_off:.3f}")
print(f"  min: {min_off:.3f},  max: {max_off:.3f}")

if mean_off > 0.55:
    verdict = "[RED] 의심: 관련 없는 쿼리 간 유사도 과도. mean pooling 오작동 또는 헤더 bloat 가능성"
elif mean_off > 0.40:
    verdict = "[YELLOW] 보통: 허용 범위지만 retrieval quality tuning 여지 있음"
else:
    verdict = "[GREEN] BGE-M3 정상 동작 (query discrimination OK)"

print(f"\n{verdict}")

## Cell 6 — 진단 요약 (자동 판정)

In [ ]:
print("=" * 60)
print(" Phase A — BGE-M3 Sanity Check — Final Diagnosis")
print("=" * 60)

pooling_str = pooling_module.get_pooling_mode_str()
dim = model.get_sentence_embedding_dimension()
max_seq = model.max_seq_length

oks, warns, errs = [], [], []

# Pooling check
if "cls" in pooling_str.lower():
    oks.append(f"[GREEN] Pooling: {pooling_str} (BGE-M3 권장)")
elif "mean" in pooling_str.lower():
    errs.append(f"[RED] Pooling: {pooling_str} — BGE-M3 은 CLS 권장. retrieval 품질 저하 직접 원인 가능")
else:
    warns.append(f"[YELLOW] Pooling: {pooling_str} — 확인 필요")

# max_seq check
if max_seq >= 4096:
    oks.append(f"[GREEN] max_seq: {max_seq} (충분)")
elif max_seq >= 1024:
    warns.append(f"[YELLOW] max_seq: {max_seq} — 일반 chunk OK, 긴 본문 일부 truncation 가능")
elif max_seq == 512:
    errs.append(f"[RED] max_seq: 512 — 본문 silent truncation 심각. BGE-M3 원본은 8192")
else:
    errs.append(f"[RED] max_seq: {max_seq} — 비정상 값, 재설정 필요")

# Dim check
if dim == 1024:
    oks.append(f"[GREEN] embedding dim: {dim} (ChromaDB collection 호환)")
else:
    errs.append(f"[RED] embedding dim: {dim} — 기대값 1024, collection 재생성 필요")

# Discrimination check
if mean_off < 0.40:
    oks.append(f"[GREEN] query discrimination: avg {mean_off:.3f} (양호)")
elif mean_off < 0.55:
    warns.append(f"[YELLOW] query discrimination: avg {mean_off:.3f} (경계선)")
else:
    errs.append(f"[RED] query discrimination: avg {mean_off:.3f} (과도한 similarity, 품질 저하)")

print()
for x in oks:   print(x)
for x in warns: print(x)
for x in errs:  print(x)

print()
print("=" * 60)
if errs:
    print("[RED] Phase A: FAIL — 수정 필요 항목이 있어 Phase B(임베딩 재구축) 전 조치가 필요합니다.")
    print("   조치 방법은 Claude에게 이 출력 전체를 복사해서 전달해주세요.")
elif warns:
    print("[YELLOW] Phase A: CONDITIONAL PASS — 경미한 경고는 있지만 Phase B 진행 가능합니다.")
else:
    print("[GREEN] Phase A: PASS — BGE-M3 세팅 건전. Phase B(임베딩 재구축)로 진행 가능합니다.")
print("=" * 60)